In [1]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [2]:
!wget -q https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task

In [2]:
import mediapipe as mp
import numpy as np

mp_hands = mp.tasks.vision.HandLandmarksConnections
mp_drawing = mp.tasks.vision.drawing_utils
mp_drawing_styles = mp.tasks.vision.drawing_styles

MARGIN = 10  # pixels
FONT_SIZE = 1
FONT_THICKNESS = 1
HANDEDNESS_TEXT_COLOR = (88, 205, 54) # vibrant green

def draw_landmarks_on_image(rgb_image, detection_result):
  hand_landmarks_list = detection_result.hand_landmarks
  handedness_list = detection_result.handedness
  annotated_image = np.copy(rgb_image)

  # Loop through the detected hands to visualize.
  for idx in range(len(hand_landmarks_list)):
    hand_landmarks = hand_landmarks_list[idx]
    handedness = handedness_list[idx]

    # Draw the hand landmarks.
    mp_drawing.draw_landmarks(
      annotated_image,
      hand_landmarks,
      mp_hands.HAND_CONNECTIONS,
      mp_drawing_styles.get_default_hand_landmarks_style(),
      mp_drawing_styles.get_default_hand_connections_style())

    # Get the top left corner of the detected hand's bounding box.
    height, width, _ = annotated_image.shape
    x_coordinates = [landmark.x for landmark in hand_landmarks]
    y_coordinates = [landmark.y for landmark in hand_landmarks]
    text_x = int(min(x_coordinates) * width)
    text_y = int(min(y_coordinates) * height) - MARGIN

    # Draw handedness (left or right hand) on the image.
    cv2.putText(annotated_image, f"{handedness[0].category_name}",
                (text_x, text_y), cv2.FONT_HERSHEY_DUPLEX,
                FONT_SIZE, HANDEDNESS_TEXT_COLOR, FONT_THICKNESS, cv2.LINE_AA)

  return annotated_image

In [ ]:
# STEP 2: Create an HandLandmarker object.
base_options = python.BaseOptions(model_asset_path='hand_landmarker.task')
options = vision.HandLandmarkerOptions(base_options=base_options,
                                       num_hands=1)
detector = vision.HandLandmarker.create_from_options(options)





cap =cv2.VideoCapture(0)
cap.set(3,1200)
cap.set(4,1000)


while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Flip the frame horizontally for a mirrored view
    frame = cv2.flip(frame, 1)
    ten_percent_box_width = int((frame.shape[1]//2) * 0.01)
    left_side_box = [0+ten_percent_box_width,(frame.shape[1] // 2-ten_percent_box_width)]
    right_side_box = [(frame.shape[1] // 2)+ten_percent_box_width, frame.shape[1]-ten_percent_box_width]
    
    
    

    image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)
    detection_result = detector.detect(image)


    

    
    #left box


    annotated_image = draw_landmarks_on_image(image.numpy_view(), detection_result)

    y1=0
    y2=frame.shape[0]
        
    
    cv2.rectangle(
                    annotated_image,
                    (left_side_box[0], y1),
                    (left_side_box[1], y2),
                    (0, 255, 0),
                    thickness=10
                )
           
    cv2.rectangle(
                annotated_image,
                (right_side_box[0], y1),
                (right_side_box[1], y2),
                (255, 0, 0),
                thickness=10
            )

    detected_hands = detection_result.hand_landmarks

    MIDDLE_FINGER_MCP_9 = detected_hands[0][8] if detected_hands else None
    MIDDLE_FINGER_PIP_10 = detected_hands[0][9] if detected_hands else None
    RING_FINGER_MCP_13 = detected_hands[0][12] if detected_hands else None
    RING_FINGER_PIP_14 =  detected_hands[0][13] if detected_hands else None

    if MIDDLE_FINGER_MCP_9 and MIDDLE_FINGER_PIP_10 and RING_FINGER_MCP_13 and RING_FINGER_PIP_14:
        HAND_AVG_X_CORD = np.mean([MIDDLE_FINGER_MCP_9.x, MIDDLE_FINGER_PIP_10.x, RING_FINGER_MCP_13.x, RING_FINGER_PIP_14.x])
        HAND_AVG_Y_CORD = np.mean([MIDDLE_FINGER_MCP_9.y, MIDDLE_FINGER_PIP_10.y, RING_FINGER_MCP_13.y, RING_FINGER_PIP_14.y])

        # Display the angle on the image
        cv2.putText(annotated_image, f"Hand Avg X: {HAND_AVG_X_CORD:.2f}, Hand Avg Y: {HAND_AVG_Y_CORD:.2f}",
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)


        self.handle_paddle_movement(HAND_AVG_Y_CORD,0)


    # Display the frame
    cv2.imshow('Hand Tracking',annotated_image )
    # Break the loop on 'q' key press
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

    # Release the camera and close all OpenCV windows after the loop
    cap.release()
    cv2.destroyAllWindows()



317 0
317 0
307 0
310 0
310 0
313 0
313 0
311 0
311 0
311 0
308 0
308 0
300 0
300 0
291 0
276 0
276 0
248 0
248 0
220 0
199 0
199 0
207 0
219 0
219 0
236 0
236 0
236 0
262 0
262 0
298 0
298 0
330 0
330 0
425 0
425 0
406 0
406 0
384 0
361 0
361 0
361 0
342 0
342 0
326 0
326 0
310 0
310 0
293 0
293 0
273 0
250 0
250 0
250 0
231 0
231 0
213 0
213 0
201 0
201 0
189 0
189 0
174 0
174 0
159 0
159 0
145 0
145 0
133 0
133 0
122 0
122 0
111 0
111 0
106 0
106 0
98 0
98 0
90 0
90 0
61 0
61 0
133 0
133 0
165 0
165 0
188 0
188 0
215 0
215 0
244 0
244 0
267 0
267 0
290 0
290 0


In [ ]:
def handle_paddle_movement(left_paddle, right_paddle,left_hand_pos,right_hand_pos):

    left_paddle_mapped =  int(np.interp(left_hand_pos, [0.1, 0.8], [0, 500]))

    right_paddle_mapped = int(np.interp(right_hand_pos, [0.1, 0.8], [0, 500]))




    if (left_paddle.y - left_paddle.VEL >= 0) and (left_paddle_mapped<=left_paddle.y)  :
        left_paddle.move(up=True)
    if (left_paddle.y + left_paddle.VEL + left_paddle.height <= HEIGHT) and (left_paddle_mapped>=left_paddle.y) :
        left_paddle.move(up=False)

    if (right_paddle.y - right_paddle.VEL >= 0) and (right_paddle_mapped<=right_paddle.y)  :
            right_paddle.move(up=True)
    if (right_paddle.y + right_paddle.VEL + right_paddle.height <= HEIGHT) and (right_paddle_mapped>=right_paddle.y) :
            right_paddle.move(up=False)

            

In [17]:
# Dummy values for local notebook testing
HEIGHT = 600

def handle_paddle_movement(left_hand_y_pos, right_hand_y_pos):
    # Test values: map normalized hand positions to paddle space
    left_paddle_mapped = int(np.interp(left_hand_y_pos, [0, 1], [0, HEIGHT]))
    right_paddle_mapped = int(np.interp(right_hand_y_pos, [0, 1], [0, HEIGHT]))

    # Use the mapped values in the checks to avoid "unused variable" warnings
    left_target = left_paddle_mapped
    right_target = right_paddle_mapped

    print(left_target,right_target)